***Project 5 W-Shape Tension Member Automation
W-Shaped tension member tensile yielding checks.***

***W-Shape_Table.csv and ASTM_Material_Fy_Table need to be a folder above the .ipynb file for the filepath to work***

Import Python Libaries.

In [123]:
from dataclasses import dataclass
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

***Constants.***

In [124]:
PHI_YIELDING = 0.90
OMEGA_YIELDING = 1.67
DEFAULT_ASTM = "A992" #Most common

In [125]:
#Dataclass to validate strings and floats in the csv files
@dataclass
class DesignInput:
    scenario: str
    shape: str
    material: str
    dead_load_kips: float
    live_load_kips: float

In [126]:
#shapes_df = pd.read_csv("W_Shape_Table.csv")
#astm_df = pd.read_csv("W_Shape_Table.csv"
print(shapes_df.head())

  Type EDI_Std_Nomenclature AISC_Manual_Label T_F      W      A     d  \
0    W              W44X408           W44X408   T  408.0  120.0  44.8   
1    W              W44X368           W44X368   F  368.0  108.0  44.4   
2    W              W44X335           W44X335   F  335.0   98.5  44.0   
3    W              W44X290           W44X290   F  290.0   85.4  43.6   
4    W              W44X262           W44X262   F  262.0   77.2  43.3   

       ddet Ht  h  ...  rts.1    ho.1    PA.1 PA2.1    PB.1    PC.1    PD.1  \
0  44  3/4   –  –  ...  110.0  1080.0  3400.0     –  3810.0  2690.0  3100.0   
1  44  3/8   –  –  ...  109.0  1080.0  3380.0     –  3780.0  2670.0  3070.0   
2  44        –  –  ...  108.0  1070.0  3350.0     –  3760.0  2640.0  3050.0   
3  43  5/8   –  –  ...  107.0  1070.0  3330.0     –  3730.0  2620.0  3020.0   
4  43  1/4   –  –  ...  106.0  1060.0  3330.0     –  3730.0  2590.0  3000.0   

     T.1  WGi.1  WGo.1  
0  965.0  140.0   76.2  
1  965.0  140.0   76.2  
2  965.0  1

In [127]:
print(astm_df.head())

  Type EDI_Std_Nomenclature AISC_Manual_Label T_F      W      A     d  \
0    W              W44X408           W44X408   T  408.0  120.0  44.8   
1    W              W44X368           W44X368   F  368.0  108.0  44.4   
2    W              W44X335           W44X335   F  335.0   98.5  44.0   
3    W              W44X290           W44X290   F  290.0   85.4  43.6   
4    W              W44X262           W44X262   F  262.0   77.2  43.3   

       ddet Ht  h  ...  rts.1    ho.1    PA.1 PA2.1    PB.1    PC.1    PD.1  \
0  44  3/4   –  –  ...  110.0  1080.0  3400.0     –  3810.0  2690.0  3100.0   
1  44  3/8   –  –  ...  109.0  1080.0  3380.0     –  3780.0  2670.0  3070.0   
2  44        –  –  ...  108.0  1070.0  3350.0     –  3760.0  2640.0  3050.0   
3  43  5/8   –  –  ...  107.0  1070.0  3330.0     –  3730.0  2620.0  3020.0   
4  43  1/4   –  –  ...  106.0  1060.0  3330.0     –  3730.0  2590.0  3000.0   

     T.1  WGi.1  WGo.1  
0  965.0  140.0   76.2  
1  965.0  140.0   76.2  
2  965.0  1

In [128]:
def tensile_yielding_check(D: float, L: float, Fy: float, Ag: float) -> dict:
    Pu = 1.2 * D + 1.6 * L #LRFD
    Pa = D + L #ASD
    Pn = Fy * Ag #Nominal Tensile Strenght
    phiPn = PHI_YIELDING * Pn #LRFD Available Strenght
    Pallow = Pn / OMEGA_YIELDING # ASD Capacity check

    u_lrfd = (Pu / phiPn) * 100 #Percent Utilisation LRFD
    u_asd = (Pa / Pallow) * 100 #Percent Utilisation ASD

    return {
        "Pu (kips)": Pu,
        "Pa (kips)": Pa,
        "Pn (kips)": Pn,
        "phi*Pn (kips)": phiPn,
        "Pn/Omega (kips)": Pallow,
        "LRFD Utilization (%)": u_lrfd,
        "ASD Utilization (%)": u_asd,
        "LRFD Pass": "PASS" if phiPn >= Pu else "FAIL",
        "ASD Pass": "PASS" if Pallow >= Pa else "FAIL",
    }

***Fix file Path issues***

In [155]:
#Read only the first AISC label and first area column from the CSV.
def load_shape_table(csv_path: str) -> pd.DataFrame:
    shapes = pd.read_csv(csv_path, usecols=[2, 5])
    shapes.columns = ["shape", "Ag"]
    shapes["shape"] = shapes["shape"].astype(str).str.strip().str.upper()
    shapes["Ag"] = pd.to_numeric(shapes["Ag"], errors="coerce")
    shapes = shapes.dropna(subset=["shape", "Ag"]).drop_duplicates(subset=["shape"])
    return shapes

In [141]:
#ASTM
def load_astm_table(csv_path: str) -> pd.DataFrame:
    astm = pd.read_csv(csv_path, usecols=["ASTM Designation", "Fy"])
    astm.columns = ["material", "Fy"]
    astm["material"] = astm["material"].astype(str).str.strip().str.upper()
    astm["Fy"] = pd.to_numeric(astm["Fy"], errors="coerce")
    astm = astm.dropna(subset=["material", "Fy"]).drop_duplicates(subset=["material"])
    return astm

In [142]:
#Find gross area Ag from W_Shape_Table.csv.
def get_ag(shape: str, shapes_df: pd.DataFrame) -> float:
    shape_clean = shape.strip().upper()
    row = shapes_df[shapes_df["shape"] == shape_clean]
    if row.empty:
        raise ValueError(f"Shape '{shape}' not found in W_Shape_Table.csv")
    return float(row.iloc[0]["Ag"])

In [143]:
#Use A992 from example when material input is blank.
def get_fy(material: str, astm_df: pd.DataFrame) -> tuple[str, float]:
    material_clean = material.strip().upper()
    if material_clean == "":
        material_clean = DEFAULT_ASTM

    row = astm_df[astm_df["material"] == material_clean]
    if row.empty:
        raise ValueError(f"Material '{material_clean}' not found in ASTM_Material_Fy_Table.csv")
    return material_clean, float(row.iloc[0]["Fy"])

In [176]:
#Run one scenario and return one row of results.
def run_case(case: DesignInput, shapes_df: pd.DataFrame, astm_df: pd.DataFrame) -> dict:
    material_used, Fy = get_fy(case.material, astm_df)
    Ag = get_ag(case.shape, shapes_df)
    result = tensile_yielding_check(case.dead_load_kips, case.live_load_kips, Fy, Ag)

    return {
        "Scenario": case.scenario,
        "Shape": case.shape.strip().upper(),
        "Material": material_used,
        "Fy (ksi)": Fy,
        "Ag (in^2)": Ag,
        "D (kips)": case.dead_load_kips,
        "L (kips)": case.live_load_kips,
        **result,
    }

In [145]:
#Run all scenarios and combine into one table.
def evaluate_cases(cases: list[DesignInput], shapes_df: pd.DataFrame, astm_df: pd.DataFrame) -> pd.DataFrame:
    rows = [run_case(case, shapes_df, astm_df) for case in cases]
    return pd.DataFrame(rows)

In [146]:
#Use relative paths ***Have in correct location***
shape_table_path = "../W_Shape_Table.csv"

astm_table_path = "../ASTM_Material_Fy_Table.csv"

#Load source tables after relative paths
shapes_df = load_shape_table(shape_table_path)

astm_df = load_astm_table(astm_table_path)

print(f"Loaded {len(shapes_df)} shapes from W_Shape_Table.csv")
print(f"Loaded {len(astm_df)} materials from ASTM_Material_Fy_Table.csv")

Loaded 289 shapes from W_Shape_Table.csv
Loaded 13 materials from ASTM_Material_Fy_Table.csv


In [147]:
#Ask for a valid W-shape.
def prompt_shape() -> str:
    while True:
        shape = input("Enter W-shape designation (example W12X50): ").strip()
        try:
            _ = get_ag(shape, shapes_df)
            return shape
        except ValueError:
            print("Shape not found. Try again.")

In [148]:
#Blank input defaults to A992

def prompt_material() -> str:
    while True:
        material = input("Enter ASTM material (blank defaults to A992): ").strip()
        try:
            material_used, _ = get_fy(material, astm_df)
            return material_used
        except ValueError:
            print("Material not found. Try again (example: A36 or A572 Gr.50).")

In [149]:
#Ask for a nonnegative number.

def prompt_nonnegative_float(label: str) -> float:
    while True:
        try:
            value = float(input(label).strip())
            if value < 0:
                print("Value must be a positive number.")
                continue
            return value
        except ValueError:
            print("Please enter a number.")

***From Example D=130, L=140***

In [163]:
#Collect user scenario through sequential inputs.
user_case = DesignInput(
    scenario="User Input",
    shape=prompt_shape(),
    material=prompt_material(),
    dead_load_kips=prompt_nonnegative_float("Enter dead load D (kips): "),
    live_load_kips=prompt_nonnegative_float("Enter live load L (kips): "),
)

Enter W-shape designation (example W12X50):  W12X50
Enter ASTM material (blank defaults to A992):  
Enter dead load D (kips):  130
Enter live load L (kips):  140


In [172]:
#Three comparison scenarios from .csv values
scenario_1 = DesignInput("Scenario 1", "W14X90", "A572 Gr.50", 150.0, 95.0)

scenario_2 = DesignInput("Scenario 2", "W10X30", "A36", 90.0, 60.0)

scenario_3 = DesignInput("Scenario 3", "W16X57", "A992", 5000, 6000) #Fail

In [173]:
#Run calculations.
results = evaluate_cases([user_case, scenario_1, scenario_2, scenario_3], shapes_df, astm_df)

In [174]:
formatted = results.copy()
num_cols = formatted.select_dtypes(include="number").columns
formatted[num_cols] = formatted[num_cols].round(2)

In [175]:
show_cols = [
    "Scenario", "Shape", "Material", "Ag (in^2)", "Fy (ksi)",
    "Pu (kips)", "Pa (kips)", "phi*Pn (kips)", "Pn/Omega (kips)",
    "LRFD Utilization (%)", "ASD Utilization (%)", "LRFD Pass", "ASD Pass"
]

print(formatted[show_cols].to_string(index=False))
results

  Scenario  Shape   Material  Ag (in^2)  Fy (ksi)  Pu (kips)  Pa (kips)  phi*Pn (kips)  Pn/Omega (kips)  LRFD Utilization (%)  ASD Utilization (%) LRFD Pass ASD Pass
User Input W12X50       A992      14.60      50.0      380.0      270.0         657.00           437.13                 57.84                61.77      PASS     PASS
Scenario 1 W14X90 A572 GR.50      26.50      50.0      332.0      245.0        1192.50           793.41                 27.84                30.88      PASS     PASS
Scenario 2 W10X30        A36       8.84      32.0      204.0      150.0         254.59           169.39                 80.13                88.55      PASS     PASS
Scenario 3 W16X57       A992      16.80      50.0    15600.0    11000.0         756.00           502.99               2063.49              2186.90      FAIL     FAIL


,Scenario,Shape,Material,Fy (ksi),Ag (in^2),D (kips),L (kips),Pu (kips),Pa (kips),Pn (kips),phi*Pn (kips),Pn/Omega (kips),LRFD Utilization (%),ASD Utilization (%),LRFD Pass,ASD Pass
0,User Input,W12X50,A992,50.0,14.60,130.0,140.0,380.0,270.0,730.00,657.000,437.125749,57.838661,61.767123,PASS,PASS
1,Scenario 1,W14X90,A572 GR.50,50.0,26.50,150.0,95.0,332.0,245.0,1325.00,1192.500,793.413174,27.840671,30.879245,PASS,PASS
2,Scenario 2,W10X30,A36,32.0,8.84,90.0,60.0,204.0,150.0,282.88,254.592,169.389222,80.128205,88.553450,PASS,PASS
3,Scenario 3,W16X57,A992,50.0,16.80,5000.0,6000.0,15600.0,11000.0,840.00,756.000,502.994012,2063.492063,2186.904762,FAIL,FAIL
